# Load DIV30 Raw 10x Samples to AnnData

This notebook starts by building one combined raw `AnnData` object from the six selected DIV30 samples:

- `9853-MW-1` / `H9_rep1`
- `9853-MW-2` / `H9_rep2`
- `9853-MW-3` / `79B_rep1`
- `9853-MW-4` / `79B_rep2`
- `9853-MW-5` / `2E_rep1`
- `9853-MW-6` / `2E_rep2`

These are technical/biological replicate samples from the same omic modality. The object is therefore a within-DIV30 combined dataset, not a DIV30-vs-DIV90 comparison dataset.

Analysis keys for this object:

- `biology_key = None`: there is no biological comparison column inside this object yet.
- `batch_key = "run_sample_id"`: each 10x run sample should be treated as the batch/source sample for QC diagnostics or later batch correction.

The in-notebook class below keeps the object creation logic in one place while it is still being developed. Once the workflow is stable, this class can be moved into a `.py` module.


## Combination Logic

Each selected sample is read as a separate 10x matrix, then cells are stacked into one `AnnData` object. This means:

- cells are observations (`adata.obs`)
- genes are variables (`adata.var`)
- sample-level metadata is copied into every cell from that sample
- 10x barcodes are prefixed with `run_sample_id` before combining, because raw 10x barcode strings can repeat across samples
- genes are joined with an outer join and missing genes are filled with zero

For the six-sample DIV30 object, `run_sample_id` is the important grouping key. `biological_label` is retained as sample metadata, but the current object does not define a within-object biology contrast.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence, Tuple

import anndata as ad
import pandas as pd
import scanpy as sc


In [ ]:
@dataclass
class Raw10xAnnDataBuilder:
    """Create raw 10x AnnData objects from the DIV30/DIV90 sample map.

    This class is intentionally notebook-local for now. It is meant to make the
    loading logic explicit while we decide what belongs in a reusable module.

    Current intended use:
    - build one DIV30-only object from the first six selected run samples
    - keep all cells from those samples in one combined AnnData
    - use `run_sample_id` as the batch/sample key
    - leave `biology_key` as None because this object does not compare DIV30 vs DIV90

    The same methods can later be reused for DIV90 by changing `target_divs` and
    `target_run_sample_ids`, but this notebook currently focuses on DIV30.
    """

    project_root: Path
    target_divs: Sequence[str] = ("DIV30",)
    target_run_sample_ids: Optional[Sequence[str]] = None
    sample_map_name: str = "metadata/div30_div90_sample_id_to_biolabel_map.tsv"
    strict_missing_matrix_dirs: bool = True
    biology_key: Optional[str] = None
    batch_key: str = "run_sample_id"

    @property
    def sample_map_tsv(self) -> Path:
        """Path to the project sample map used to locate each per-sample matrix."""
        return self.project_root / self.sample_map_name

    def sample_table(self) -> pd.DataFrame:
        """Return selected sample metadata plus each sample's 10x matrix path.

        This method does not read expression data. It only prepares and validates
        the table that tells the builder which samples to load.

        Important checks performed here:
        - required metadata columns must exist
        - requested DIV values must be present
        - requested run sample IDs must be present
        - sample order follows `target_run_sample_ids` when a specific subset is given

        For the current DIV30 object, this should return exactly six rows:
        9853-MW-1 through 9853-MW-6.
        """
        sample_map = pd.read_csv(self.sample_map_tsv, sep="\t")
        required = {"DIV", "run_sample_id", "biological_label", "per_sample_metrics_csv"}
        missing = required.difference(sample_map.columns)
        if missing:
            raise ValueError(f"Sample map is missing required columns: {sorted(missing)}")

        # First restrict by time point. In this notebook that should be DIV30 only.
        sample_map = sample_map[sample_map["DIV"].isin(self.target_divs)].copy()

        # Then optionally restrict to the exact run samples we want in this object.
        # This is how we exclude DIV30 samples 9853-MW-7, 9853-MW-8, and 9853-MW-9.
        if self.target_run_sample_ids is not None:
            sample_map = sample_map[sample_map["run_sample_id"].isin(self.target_run_sample_ids)].copy()
            found = set(sample_map["run_sample_id"].astype(str))
            missing_ids = [sid for sid in self.target_run_sample_ids if sid not in found]
            if missing_ids:
                raise ValueError(f"Missing requested run_sample_id values in sample map: {missing_ids}")

        if sample_map.empty:
            raise ValueError("No samples matched target_divs/target_run_sample_ids.")

        # Ordered categoricals keep displays and summaries in the requested order.
        sample_map["DIV"] = pd.Categorical(sample_map["DIV"], categories=self.target_divs, ordered=True)
        if self.target_run_sample_ids is not None:
            sample_map["run_sample_id"] = pd.Categorical(
                sample_map["run_sample_id"],
                categories=self.target_run_sample_ids,
                ordered=True,
            )

        sample_map = sample_map.sort_values(["DIV", "run_sample_id"]).reset_index(drop=True)

        # Cell Ranger per-sample outputs store the filtered matrix next to metrics_summary.csv.
        sample_map["matrix_dir"] = sample_map["per_sample_metrics_csv"].map(
            lambda p: str(Path(p).parent / "count" / "sample_filtered_feature_bc_matrix")
        )
        return sample_map

    def read_one_sample(self, row: pd.Series) -> ad.AnnData:
        """Read one per-sample 10x matrix and add sample metadata to every cell.

        The matrix is read with gene symbols as `var_names`. Cell barcodes are
        prefixed with the sample ID before combining, for example:

        `9853-MW-1:AAACAAGCAAGATAAGACTTTAGG-1`

        This matters because the original 10x barcode suffixes can repeat across
        run samples. Prefixing prevents duplicated `.obs_names` in the combined
        object and makes it obvious which sample each cell came from.
        """
        matrix_dir = Path(row["matrix_dir"])
        if not matrix_dir.exists():
            raise FileNotFoundError(f"Missing per-sample 10x matrix directory: {matrix_dir}")

        one = sc.read_10x_mtx(matrix_dir, var_names="gene_symbols", make_unique=True)
        run_sample_id = str(row["run_sample_id"])

        one.obs_names = [f"{run_sample_id}:{barcode}" for barcode in one.obs_names]
        one.obs["DIV"] = str(row["DIV"])
        one.obs["run_sample_id"] = run_sample_id
        one.obs["biological_label"] = str(row["biological_label"])
        one.obs["matrix_dir"] = str(matrix_dir)
        return one

    def component_anndatas(self):
        """Return `(run_sample_id, AnnData)` pairs for selected samples.

        This format mirrors the input expected by `snapatac2.AnnDataSet`, where
        each component object has a key. It also gives us a useful intermediate
        checkpoint before building the combined object.

        Missing matrix directories are treated as an error by default so a partial
        dataset is not accidentally analyzed as if it were complete.
        """
        components = []
        missing_dirs = []

        for _, row in self.sample_table().iterrows():
            matrix_dir = Path(row["matrix_dir"])
            if not matrix_dir.exists():
                missing_dirs.append(str(matrix_dir))
                continue
            components.append((str(row["run_sample_id"]), self.read_one_sample(row)))

        if missing_dirs:
            message = "Missing per-sample 10x matrix directories:\n" + "\n".join(f" - {p}" for p in missing_dirs)
            if self.strict_missing_matrix_dirs:
                raise FileNotFoundError(message)
            print(message)

        if not components:
            raise FileNotFoundError("No per-sample 10x matrix directories were found.")

        return components

    def combined_anndata(self) -> ad.AnnData:
        """Build one standard AnnData by stacking cells from selected samples.

        This is the main output for Scanpy work in this notebook. It concatenates
        along observations (`axis=0`), so the result is one cell-by-gene matrix.

        `join="outer"` keeps the union of genes across samples. `fill_value=0`
        is appropriate for count matrices because a gene absent from one sample's
        matrix should be represented as zero counts after alignment.

        The returned object records the analysis keys in `.uns`:
        - `.uns["biology_key"] = None`
        - `.uns["batch_key"] = "run_sample_id"`
        """
        sample_map = self.sample_table()
        combined = ad.concat(
            [one for _, one in self.component_anndatas()],
            axis=0,
            join="outer",
            merge="same",
            fill_value=0,
            index_unique=None,
        )
        self._finalize_obs(combined, sample_map)
        combined.uns["sample_map_tsv"] = str(self.sample_map_tsv)
        combined.uns["target_divs"] = list(self.target_divs)
        combined.uns["biology_key"] = self.biology_key
        combined.uns["batch_key"] = self.batch_key
        combined.uns["combine_logic"] = (
            "Per-sample 10x matrices were read separately, annotated, barcode-prefixed by "
            "run_sample_id, and concatenated along observations with an outer gene join."
        )
        return combined

    def anndata_set(self, filename: Path, add_key: str = "run_sample_id"):
        """Build a backed snapatac2 AnnDataSet from selected per-sample objects.

        This is optional. Use it only if you want a `.h5ads` file that preserves
        access to component AnnData objects. For routine Scanpy analysis, the
        standard `combined_anndata()` output is the simpler first target.
        """
        try:
            import snapatac2 as snap
        except ModuleNotFoundError as exc:
            raise ModuleNotFoundError("Install snapatac2 to use AnnDataSet output.") from exc

        filename = Path(filename)
        filename.parent.mkdir(parents=True, exist_ok=True)
        return snap.AnnDataSet(
            adatas=self.component_anndatas(),
            filename=str(filename),
            add_key=add_key,
        )

    def cell_count_tables(self, adata: ad.AnnData) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Return QC summaries that confirm the combined object composition.

        `sample_counts` should show one row per selected DIV30 run sample.
        `div_counts` should show one DIV row because this object is DIV30 only.
        """
        sample_counts = (
            adata.obs.groupby(["DIV", "run_sample_id", "biological_label"], observed=True)
            .size()
            .rename("n_cells")
            .reset_index()
            .sort_values(["DIV", "run_sample_id"])
            .reset_index(drop=True)
        )
        div_counts = adata.obs.groupby("DIV", observed=True).size().rename("n_cells").reset_index()
        return sample_counts, div_counts

    def _finalize_obs(self, adata: ad.AnnData, sample_map: pd.DataFrame) -> None:
        """Apply final metadata types and fail if cell IDs are not unique."""
        if not adata.obs_names.is_unique:
            duplicated = adata.obs_names[adata.obs_names.duplicated()].unique()[:10].tolist()
            raise ValueError(f"Combined AnnData has duplicated obs_names. Examples: {duplicated}")

        adata.obs["DIV"] = pd.Categorical(adata.obs["DIV"], categories=self.target_divs, ordered=True)
        adata.obs["run_sample_id"] = pd.Categorical(
            adata.obs["run_sample_id"],
            categories=sample_map["run_sample_id"].astype(str).tolist(),
            ordered=True,
        )
        adata.obs["biological_label"] = adata.obs["biological_label"].astype("string")
        adata.obs["matrix_dir"] = adata.obs["matrix_dir"].astype("string")


In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from the notebook working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "metadata" / "div30_div90_sample_id_to_biolabel_map.tsv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing metadata/div30_div90_sample_id_to_biolabel_map.tsv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# The current object is intentionally limited to these six DIV30 samples.
# DIV30 samples 9853-MW-7, 9853-MW-8, and 9853-MW-9 are excluded here.
DIV30_TARGET_RUN_SAMPLE_IDS = (
    "9853-MW-1",  # H9_rep1
    "9853-MW-2",  # H9_rep2
    "9853-MW-3",  # 79B_rep1
    "9853-MW-4",  # 79B_rep2
    "9853-MW-5",  # 2E_rep1
    "9853-MW-6",  # 2E_rep2
)

# Since this is a DIV30-only object, there is no biology comparison key yet.
# The sample/run identifier is the batch key for downstream QC or correction.
builder = Raw10xAnnDataBuilder(
    project_root=PROJECT_ROOT,
    target_divs=("DIV30",),
    target_run_sample_ids=DIV30_TARGET_RUN_SAMPLE_IDS,
    biology_key=None,
    batch_key="run_sample_id",
)

sample_map = builder.sample_table()
print(f"Selected {len(sample_map)} samples from: {builder.sample_map_tsv}")
print("biology_key:", builder.biology_key)
print("batch_key:", builder.batch_key)
display(sample_map[["DIV", "run_sample_id", "biological_label", "matrix_dir"]])


In [ ]:
# Build the standard combined AnnData object for downstream Scanpy work.
# This reads each selected 10x matrix from disk, annotates cells, and stacks cells into one object.
# Expected result: one DIV30-only AnnData with six run_sample_id batches.
sample_adata = builder.combined_anndata()

print("Combined AnnData shape:", sample_adata.shape)
print("Loaded run_sample_id count:", sample_adata.obs["run_sample_id"].nunique())
print("Unique cell IDs:", sample_adata.obs_names.is_unique)
print("biology_key:", sample_adata.uns["biology_key"])
print("batch_key:", sample_adata.uns["batch_key"])


In [ ]:
# Optional: build a backed snapatac2 AnnDataSet instead of a single in-memory AnnData.
# This follows the AnnDataSet pattern you found: a list of keyed component AnnData objects.
# Leave this commented out until you specifically want `.h5ads` output and have snapatac2 installed.
# dataset = builder.anndata_set(PROJECT_ROOT / "results" / "python_anndata" / "varela_div30_six_samples_raw.h5ads")
# dataset


In [ ]:
# Multisample sanity checks: per-sample and per-DIV cell totals.
sample_cell_counts, div_cell_counts = builder.cell_count_tables(sample_adata)

print("obs columns:", list(sample_adata.obs.columns))
print("Samples loaded:", sample_cell_counts["run_sample_id"].nunique())
display(sample_cell_counts)
display(div_cell_counts)


In [ ]:
# Quick object preview
sample_adata


In [ ]:
# Compute Scanpy QC metrics for the combined object.
sample_adata.var["mt"] = sample_adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    sample_adata,
    qc_vars=["mt"],
    percent_top=[20],
    log1p=True,
    inplace=True,
)

qc_report = pd.Series(
    {
        "n_cells": int(sample_adata.n_obs),
        "n_genes": int(sample_adata.n_vars),
        "n_samples": int(sample_adata.obs["run_sample_id"].nunique()),
        "divs": ",".join(sample_adata.obs["DIV"].dropna().astype(str).unique()),
        "median_total_counts": float(sample_adata.obs["total_counts"].median()),
        "median_n_genes_by_counts": float(sample_adata.obs["n_genes_by_counts"].median()),
        "median_pct_counts_mt": float(sample_adata.obs["pct_counts_mt"].median()),
    },
    name="value",
)
display(qc_report.to_frame())


In [ ]:
# Minimal metadata previews used in QC reporting
display(sample_adata.obs[["DIV", "run_sample_id", "biological_label"]].head())
display(sample_adata.var.head())


In [ ]:
# QC plots (Scanpy): distributions of counts, genes, and mitochondrial fraction.
sc.pl.violin(
    sample_adata,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:
sc.pl.scatter(sample_adata, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt")


In [ ]:
# Optional: export per-cell QC values for records.
qc_out = PROJECT_ROOT / "results" / "python_anndata" / "qc_report_div30_six_samples_raw_multisample.tsv"
qc_out.parent.mkdir(parents=True, exist_ok=True)
sample_adata.obs[[
    "DIV",
    "run_sample_id",
    "biological_label",
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
]].to_csv(qc_out, sep="	", index=True)
print("Wrote", qc_out)
